In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

In [4]:
np.random.seed(42)
tf.random.set_seed(42)

In [5]:
train = pd.read_parquet('../data/cleanedTrain.parquet')

In [6]:
train.sort_values('timestamp', inplace=True)

In [7]:
train.replace([np.inf, -np.inf], np.nan, inplace=True)
train.fillna(train.median(), inplace=True)

In [8]:
feature_cols = [col for col in train.columns if col not in ['timestamp', 'label']]
print(train[feature_cols].dtypes.value_counts())

scaler = StandardScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols]).astype(np.float32)
train['label'] = train['label'].astype(np.float32)
print(train[feature_cols].dtypes.value_counts())

float64    895
Name: count, dtype: int64


d:\Anaconda\Lib\site-packages\sklearn\utils\extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
d:\Anaconda\Lib\site-packages\sklearn\utils\extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
d:\Anaconda\Lib\site-packages\sklearn\utils\extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


float32    895
Name: count, dtype: int64


In [9]:
print(train[feature_cols].isnull().sum().sum())


11043627


In [10]:
train[feature_cols] = train[feature_cols].fillna(0)


In [11]:
X = train[feature_cols]
y = train['label']

selector = SelectKBest(score_func=f_regression, k=250)
X_selected = selector.fit_transform(X, y)

selected_features = [feature_cols[i] for i in selector.get_support(indices=True)]
feature_cols = selected_features
print(f"Selected {len(feature_cols)} features")

Selected 250 features


In [12]:
train = train.copy()


In [13]:
train['volume_bin'], bin_edges = pd.qcut(train['volume'], q=3, labels=['low', 'mid', 'high'], retbins=True)

In [14]:
def create_sequences(df, timesteps=10):
    Xs, ys = [], []
    for i in range(len(df) - timesteps):
        seq = df[feature_cols].iloc[i:i+timesteps].values.astype(np.float32)
        print(seq.shape)  
        Xs.append(seq)
        ys.append(df['label'].iloc[i+timesteps])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)


In [15]:
def build_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(128, input_shape=input_shape, return_sequences=False))
    model.add(Dropout(0.3))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse')
    return model


In [20]:
class CryptoSequence(Sequence):
    def __init__(self, df, features, timesteps=5, batch_size=256, **kwargs):
        super().__init__(**kwargs)
        self.timesteps = timesteps
        self.batch_size = batch_size
        self.features = features
        
        self.X = df[features].values.astype(np.float32)
        self.y = df['label'].values.astype(np.float32)
        
        self.indexes = np.arange(len(self.X) - timesteps)

    def __len__(self):
        return int(np.floor(len(self.indexes) / self.batch_size))

    def __getitem__(self, idx):
        batch_start = idx * self.batch_size
        batch_end = batch_start + self.batch_size
        batch_indexes = self.indexes[batch_start:batch_end]

        X_batch = np.array([
            self.X[i:i+self.timesteps] for i in batch_indexes
        ])
        y_batch = self.y[batch_indexes + self.timesteps]

        return X_batch, y_batch

    def on_epoch_end(self):
        np.random.shuffle(self.indexes)


In [29]:
timesteps = 3
scores = {}

for bin_label in ['low', 'mid', 'high']:
    print(f"\n📊 Training for volume bin: {bin_label.upper()}")
    
    df_bin = train[train['volume_bin'] == bin_label].reset_index(drop=True)
    
    if len(df_bin) < timesteps + 100:
        print("❗ Skipped — not enough data")
        continue
    
    split_idx = int(0.8 * (len(df_bin) - timesteps))
    train_df, val_df = df_bin.iloc[:split_idx+timesteps], df_bin.iloc[split_idx:]
    
    train_gen = CryptoSequence(train_df, feature_cols, timesteps=timesteps, batch_size=512)
    val_gen = CryptoSequence(val_df, feature_cols, timesteps=timesteps, batch_size=512)
    
    model = build_lstm((timesteps, len(feature_cols)))

    
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=10,
        callbacks=[early_stop],
        verbose=2    
    )
    
    
    y_val, y_pred = [], []
    for X_batch, y_batch in val_gen:
        if len(X_batch) == 0:
            continue
        preds = model.predict(X_batch).flatten()
        y_val.extend(y_batch)
        y_pred.extend(preds)
    
    y_val = np.array(y_val)
    y_pred = np.array(y_pred)

    mask = np.isfinite(y_val) & np.isfinite(y_pred)
    corr, _ = pearsonr(y_val[mask], y_pred[mask])
    print(f"Validation Pearson: {corr:.5f}")

    
    model.save(f'../results/LSTM/LSTM_{bin_label}_volume.keras')
    scores[bin_label] = corr



📊 Training for volume bin: LOW
Epoch 1/10


d:\Anaconda\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


273/273 - 9s - 33ms/step - loss: 0.3543 - val_loss: 0.7595
Epoch 2/10
273/273 - 6s - 23ms/step - loss: 0.2189 - val_loss: 0.8648
Epoch 3/10
273/273 - 6s - 23ms/step - loss: 0.1627 - val_loss: 0.9061
Epoch 4/10
273/273 - 9s - 32ms/step - loss: 0.1377 - val_loss: 0.8988
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
16/16 ━━━━━━━━━━━━

KeyboardInterrupt: 

In [26]:
print("std y_val:", np.std(y_val))
print("std y_pred:", np.std(y_pred))


std y_val: 0.7569411
std y_pred: 0.48462713
